# SEC EDGAR 10-Q Filing Downloader

This notebook downloads all 10-Q filings submitted to SEC EDGAR between **2026-01-01** and **2026-03-18**.

It uses the SEC EDGAR Full-Text Search API (EFTS) to enumerate every 10-Q filing in the date range, then downloads the primary HTML document for each one, saving it in a hierarchical directory structure with accompanying metadata.

**Output structure:**
```
/content/data/tickers/<TICKER>/<YYYY-MM-DD>/reports/10Q-<ticker>-<accession>.html
/content/data/tickers/<TICKER>/<YYYY-MM-DD>/reports/metadata.json
```

**SEC EDGAR compliance:** The SEC requires all programmatic access to include a descriptive `User-Agent` header identifying your organisation and contact email. Update `SEC_USER_AGENT` in Cell 2 before running.

In [ ]:
# Cell 1 — Install Dependencies
# edgartools provides a high-level Python interface to SEC EDGAR.
# requests is used for direct HTTP calls to the EFTS search API.
!pip install edgartools requests --quiet

## Cell 2 — Imports & Configuration

Set `SEC_USER_AGENT` to `"YourCompanyName your@email.com"` before running. The SEC will block requests that use a generic or missing User-Agent.

In [ ]:
# Cell 2 — Imports & Configuration

import os
import re
import json
import time
import datetime
import requests
from pathlib import Path

# ---------------------------------------------------------------------------
# USER-CONFIGURABLE CONSTANTS
# ---------------------------------------------------------------------------

# REQUIRED: Update this before running. Format: "CompanyName AdminEmail"
# Example: "Acme Research research@acme.com"
SEC_USER_AGENT = "YourCompanyName your@email.com"

# Date range for 10-Q filings (inclusive)
START_DATE = "2026-01-01"
END_DATE   = "2026-03-18"

# Form type to search
FORM_TYPE = "10-Q"

# Base output directory (Google Colab working directory)
BASE_DIR = Path("/content/data/tickers")

# SEC EDGAR API endpoints
EFTS_SEARCH_URL  = "https://efts.sec.gov/LATEST/search-index"
EFTS_BROWSE_URL  = "https://www.sec.gov/cgi-bin/browse-edgar"
EDGAR_BASE_URL   = "https://www.sec.gov"
SUBMISSIONS_URL  = "https://data.sec.gov/submissions"

# Rate-limiting: SEC allows max 10 requests/second.
# 0.15 s between requests gives ~6.7 req/s — safely under the limit.
REQUEST_DELAY_SECONDS = 0.15

# Number of results to fetch per EFTS page (max supported by EFTS is 50)
PAGE_SIZE = 50

# ---------------------------------------------------------------------------
# DERIVED CONSTANTS
# ---------------------------------------------------------------------------

HEADERS = {
    "User-Agent": SEC_USER_AGENT,
    "Accept": "application/json, text/html, */*",
    "Accept-Encoding": "gzip, deflate",
}

print(f"Configuration loaded.")
print(f"  Form type : {FORM_TYPE}")
print(f"  Date range: {START_DATE} -> {END_DATE}")
print(f"  Output dir: {BASE_DIR}")
print(f"  User-Agent: {SEC_USER_AGENT}")

if SEC_USER_AGENT == "YourCompanyName your@email.com":
    print("\n[WARNING] You are using the placeholder User-Agent.")
    print("          The SEC may rate-limit or block your requests.")
    print("          Update SEC_USER_AGENT above with your real name and email.")

## Cell 3 — Helper Functions

Utilities for safe file naming, path construction, metadata persistence, duplicate detection, and resilient HTTP downloads with retry logic.

In [ ]:
# Cell 3 — Helper Functions

def sanitize_filename(name: str) -> str:
    """
    Replace any character that is not alphanumeric, a hyphen, or an underscore
    with an underscore, then collapse runs of underscores and strip leading/
    trailing underscores.  Returns 'UNKNOWN' if the result is empty.
    """
    if not name:
        return "UNKNOWN"
    sanitized = re.sub(r"[^\w\-]", "_", str(name))
    sanitized = re.sub(r"_+", "_", sanitized).strip("_")
    return sanitized or "UNKNOWN"


def build_filing_path(ticker: str, filed_date: str, accession: str) -> Path:
    """
    Construct the full directory path for a single filing.

    Structure:
        BASE_DIR/<TICKER>/<YYYY-MM-DD>/reports/

    The date string is normalised to YYYY-MM-DD regardless of input format.
    """
    safe_ticker    = sanitize_filename(ticker).upper()
    safe_accession = sanitize_filename(accession)

    # Normalise date to YYYY-MM-DD (drop any time component)
    date_part = str(filed_date)[:10]

    return BASE_DIR / safe_ticker / date_part / "reports"


def save_metadata(directory: Path, metadata: dict) -> None:
    """
    Serialize *metadata* to JSON and write it to <directory>/metadata.json.
    The directory is created if it does not already exist.
    """
    directory.mkdir(parents=True, exist_ok=True)
    meta_path = directory / "metadata.json"
    with meta_path.open("w", encoding="utf-8") as fh:
        json.dump(metadata, fh, indent=2, default=str)


def is_duplicate(directory: Path) -> bool:
    """
    Return True if <directory>/metadata.json already exists, indicating
    the filing has been downloaded in a previous run.
    """
    return (directory / "metadata.json").exists()


def download_with_retry(
    url: str,
    headers: dict,
    max_retries: int = 3,
    backoff_base: float = 2.0,
) -> requests.Response | None:
    """
    Fetch *url* with *headers*, retrying up to *max_retries* times on
    transient errors (5xx, connection errors, timeouts).

    Each retry waits backoff_base ** attempt seconds (1, 2, 4, ...).
    A rate-limit pause of REQUEST_DELAY_SECONDS is applied before every
    request so that the SEC 10-req/sec limit is never exceeded.

    Returns the Response object on success, or None if all retries fail.
    """
    for attempt in range(max_retries):
        try:
            time.sleep(REQUEST_DELAY_SECONDS)
            response = requests.get(url, headers=headers, timeout=30)

            if response.status_code == 200:
                return response

            if response.status_code == 429:
                # Rate limited — wait longer before retrying
                wait = backoff_base ** (attempt + 2)
                print(f"    [RATE LIMIT] 429 received. Waiting {wait:.0f}s before retry {attempt + 1}/{max_retries}...")
                time.sleep(wait)
                continue

            if response.status_code >= 500:
                wait = backoff_base ** attempt
                print(f"    [SERVER ERROR] {response.status_code} for {url}. Retry {attempt + 1}/{max_retries} in {wait:.0f}s...")
                time.sleep(wait)
                continue

            # 4xx (other than 429) — not retryable
            print(f"    [HTTP {response.status_code}] {url}")
            return None

        except requests.exceptions.ConnectionError as exc:
            wait = backoff_base ** attempt
            print(f"    [CONNECTION ERROR] {exc}. Retry {attempt + 1}/{max_retries} in {wait:.0f}s...")
            time.sleep(wait)

        except requests.exceptions.Timeout:
            wait = backoff_base ** attempt
            print(f"    [TIMEOUT] {url}. Retry {attempt + 1}/{max_retries} in {wait:.0f}s...")
            time.sleep(wait)

        except Exception as exc:
            print(f"    [UNEXPECTED ERROR] {exc} — giving up on {url}")
            return None

    print(f"    [FAILED] Exhausted {max_retries} retries for {url}")
    return None


def accession_to_path(accession_number: str) -> str:
    """
    Convert an accession number in the format '0001234567-26-000001'
    (dashes included) to the path segment '0001234567/26/000001'
    used in SEC EDGAR URLs.
    """
    return accession_number.replace("-", "/")


def accession_nodash(accession_number: str) -> str:
    """Return accession number with dashes removed, e.g. '0001234567-26-000001' -> '000123456726000001'."""
    return accession_number.replace("-", "")


print("Helper functions defined.")

## Cell 4 — Fetch All 10-Q Filings

We use the **SEC EDGAR Full-Text Search System (EFTS)** to enumerate every 10-Q filing accepted during the configured date range.

The EFTS endpoint returns pages of 50 results. We paginate through all pages using the `from` offset parameter until we have retrieved every matching filing.

For each hit the EFTS result contains:
- `entity_name` — company name
- `file_num` / `entity_id` — CIK
- `period_of_report` — reporting period end date
- `file_date` — date accepted by EDGAR
- `accession_no` — unique accession number

We then resolve the ticker symbol from SEC's submissions API using the CIK.

In [ ]:
# Cell 4 — Fetch All 10-Q Filings

# ---------------------------------------------------------------------------
# Step 4a: Paginate through the EFTS search index to collect raw filing hits
# ---------------------------------------------------------------------------

def fetch_all_efts_hits(start_date: str, end_date: str, form: str = "10-Q") -> list[dict]:
    """
    Query the EFTS search-index endpoint and return every matching filing
    hit for the given date range and form type.

    Pagination is handled automatically using the 'from' offset parameter.
    Returns a list of raw hit dictionaries as returned by the API.
    """
    all_hits: list[dict] = []
    offset = 0
    total_reported = None

    print(f"Fetching {form} filings from EDGAR EFTS: {start_date} -> {end_date}")
    print("-" * 60)

    while True:
        params = {
            "q":          "\"\"",   # match-all query
            "forms":      form,
            "dateRange":  "custom",
            "startdt":    start_date,
            "enddt":      end_date,
            "from":       offset,
            "size":       PAGE_SIZE,
        }

        response = download_with_retry(EFTS_SEARCH_URL, HEADERS)
        if response is None:
            # Try with requests directly (EFTS needs query params)
            time.sleep(REQUEST_DELAY_SECONDS)
            try:
                response = requests.get(
                    EFTS_SEARCH_URL,
                    params=params,
                    headers=HEADERS,
                    timeout=30,
                )
                if response.status_code != 200:
                    print(f"  [ERROR] EFTS returned HTTP {response.status_code} at offset {offset}. Stopping pagination.")
                    break
            except Exception as exc:
                print(f"  [ERROR] EFTS request failed at offset {offset}: {exc}")
                break
        else:
            # Re-issue with params (download_with_retry doesn't accept params)
            time.sleep(REQUEST_DELAY_SECONDS)
            try:
                response = requests.get(
                    EFTS_SEARCH_URL,
                    params=params,
                    headers=HEADERS,
                    timeout=30,
                )
                if response.status_code != 200:
                    print(f"  [ERROR] EFTS returned HTTP {response.status_code} at offset {offset}. Stopping.")
                    break
            except Exception as exc:
                print(f"  [ERROR] {exc}")
                break

        try:
            data = response.json()
        except ValueError:
            print(f"  [ERROR] Non-JSON response from EFTS at offset {offset}.")
            break

        hits = data.get("hits", {}).get("hits", [])

        if total_reported is None:
            total_reported = data.get("hits", {}).get("total", {}).get("value", 0)
            print(f"  Total filings reported by EDGAR: {total_reported}")

        if not hits:
            print(f"  No more results at offset {offset}. Pagination complete.")
            break

        all_hits.extend(hits)
        print(f"  Fetched offset {offset:>6} — {len(all_hits):>6} / {total_reported} collected")

        offset += PAGE_SIZE
        if total_reported is not None and offset >= total_reported:
            print(f"  Reached total count ({total_reported}). Stopping pagination.")
            break

    print(f"\nTotal raw hits collected: {len(all_hits)}")
    return all_hits


# ---------------------------------------------------------------------------
# Step 4b: CIK -> ticker mapping cache (fetched on demand from SEC)
# ---------------------------------------------------------------------------

_ticker_cache: dict[str, str] = {}  # CIK (zero-padded 10-digit) -> ticker


def cik_to_ticker(cik: str) -> str:
    """
    Look up the primary ticker for a given CIK using the SEC submissions API.
    Returns the first ticker listed, or the CIK itself if none is found.
    Results are cached to avoid redundant requests.
    """
    cik_padded = str(cik).lstrip("0").zfill(10)
    if cik_padded in _ticker_cache:
        return _ticker_cache[cik_padded]

    url = f"{SUBMISSIONS_URL}/CIK{cik_padded}.json"
    time.sleep(REQUEST_DELAY_SECONDS)
    try:
        resp = requests.get(url, headers=HEADERS, timeout=20)
        if resp.status_code == 200:
            submission_data = resp.json()
            tickers = submission_data.get("tickers", [])
            ticker = tickers[0] if tickers else f"CIK{cik_padded}"
        else:
            ticker = f"CIK{cik_padded}"
    except Exception:
        ticker = f"CIK{cik_padded}"

    _ticker_cache[cik_padded] = ticker
    return ticker


# ---------------------------------------------------------------------------
# Step 4c: Parse raw EFTS hits into clean filing records
# ---------------------------------------------------------------------------

def parse_efts_hits(hits: list[dict]) -> list[dict]:
    """
    Transform raw EFTS hit documents into normalised filing dictionaries.

    Each output dict contains:
        ticker, company_name, filed_at, accession_number, filing_url, cik
    """
    filings: list[dict] = []
    print(f"\nResolving tickers for {len(hits)} filings (this may take a while)...")

    for i, hit in enumerate(hits):
        src = hit.get("_source", {})

        # EFTS source field names vary slightly across versions
        company_name    = src.get("entity_name") or src.get("display_names", ["UNKNOWN"])[0]
        cik             = str(src.get("entity_id") or src.get("ciks", ["0000000000"])[0]).strip()
        filed_at        = src.get("file_date") or src.get("period_of_report", "")
        accession_raw   = src.get("accession_no") or hit.get("_id", "")

        # Normalise accession number to XX-XX-XX dash format
        accession_clean = accession_raw.replace("/", "-")
        acc_nodash      = accession_nodash(accession_clean)

        # Build the EDGAR filing index URL
        cik_padded  = cik.zfill(10)
        filing_url  = (
            f"https://www.sec.gov/Archives/edgar/data/{cik_padded.lstrip('0')}"
            f"/{acc_nodash}/{accession_clean}-index.htm"
        )

        # Resolve ticker (prints nothing to avoid noise; cache makes this cheap)
        ticker = cik_to_ticker(cik)

        filings.append({
            "ticker":           ticker,
            "company_name":     company_name,
            "filed_at":         filed_at[:10] if filed_at else "",
            "accession_number": accession_clean,
            "filing_url":       filing_url,
            "cik":              cik_padded,
        })

        if (i + 1) % 50 == 0:
            print(f"  Parsed {i + 1:>6} / {len(hits)} filings...")

    print(f"  Done. {len(filings)} filing records ready.")
    return filings


# ---------------------------------------------------------------------------
# Execute the fetch
# ---------------------------------------------------------------------------

raw_hits = fetch_all_efts_hits(START_DATE, END_DATE, FORM_TYPE)
ALL_FILINGS = parse_efts_hits(raw_hits)

print(f"\nFiling enumeration complete. {len(ALL_FILINGS)} total 10-Q filings to process.")

## Cell 5 — Process & Download Filings

For each filing we:
1. Build the hierarchical output path.
2. Skip if `metadata.json` already exists (idempotent re-runs).
3. Fetch the filing index page to locate the primary 10-Q HTML document.
4. Download the HTML document.
5. Save it to disk alongside `metadata.json`.

Progress is logged to stdout with counters for downloaded, skipped, and errored filings.

In [ ]:
# Cell 5 — Process & Download Filings

def get_primary_document_url(cik: str, accession_number: str) -> str | None:
    """
    Fetch the filing index JSON from SEC EDGAR and return the URL of the
    primary document (the actual 10-Q HTML or HTM file).

    Falls back to constructing a probable URL if the index cannot be parsed.
    Returns None if no suitable document is found.
    """
    cik_stripped   = cik.lstrip("0")
    acc_nodash     = accession_nodash(accession_number)

    # The EDGAR submissions API exposes filing document lists as JSON
    index_json_url = (
        f"https://data.sec.gov/submissions/CIK{cik.zfill(10)}.json"
    )

    # Primary approach: use the filing-specific index JSON
    filing_index_json = (
        f"https://www.sec.gov/Archives/edgar/data/{cik_stripped}"
        f"/{acc_nodash}/{accession_number}-index.json"
    )

    time.sleep(REQUEST_DELAY_SECONDS)
    try:
        resp = requests.get(filing_index_json, headers=HEADERS, timeout=20)
        if resp.status_code == 200:
            index_data = resp.json()
            # The index JSON has a 'documents' list; find the primary one
            documents = index_data.get("documents", [])
            # Prefer documents marked as type '10-Q'
            for doc in documents:
                if doc.get("type", "").upper() in ("10-Q", "10Q"):
                    doc_name = doc.get("name", "")
                    if doc_name.lower().endswith((".htm", ".html")):
                        return (
                            f"https://www.sec.gov/Archives/edgar/data/{cik_stripped}"
                            f"/{acc_nodash}/{doc_name}"
                        )
            # Fall through: try any .htm document
            for doc in documents:
                doc_name = doc.get("name", "")
                if doc_name.lower().endswith((".htm", ".html")):
                    return (
                        f"https://www.sec.gov/Archives/edgar/data/{cik_stripped}"
                        f"/{acc_nodash}/{doc_name}"
                    )
    except Exception as exc:
        pass  # Fall through to the HTM index approach

    # Fallback: scrape the human-readable index page for an .htm link
    filing_index_htm = (
        f"https://www.sec.gov/Archives/edgar/data/{cik_stripped}"
        f"/{acc_nodash}/{accession_number}-index.htm"
    )
    time.sleep(REQUEST_DELAY_SECONDS)
    try:
        resp = requests.get(filing_index_htm, headers={**HEADERS, "Accept": "text/html"}, timeout=20)
        if resp.status_code == 200:
            # Simple regex extraction — no BeautifulSoup dependency required
            matches = re.findall(
                r'href="(/Archives/edgar/data/[^"]+\.htm[l]?)"',
                resp.text,
                re.IGNORECASE,
            )
            # Exclude the index page itself
            for match in matches:
                if "-index" not in match.lower():
                    return f"https://www.sec.gov{match}"
    except Exception:
        pass

    return None


def process_filing(filing: dict) -> str:
    """
    Download a single filing and save it with its metadata.

    Returns one of: 'downloaded', 'skipped', 'error'
    """
    ticker           = filing["ticker"]
    filed_at         = filing["filed_at"]
    accession_number = filing["accession_number"]
    cik              = filing["cik"]

    output_dir = build_filing_path(ticker, filed_at, accession_number)

    # Duplicate check
    if is_duplicate(output_dir):
        return "skipped"

    # Resolve the URL of the primary document
    doc_url = get_primary_document_url(cik, accession_number)
    if doc_url is None:
        print(f"  [NO DOC] {ticker} | {accession_number} — could not locate primary HTML document.")
        return "error"

    # Download the HTML
    resp = download_with_retry(doc_url, {**HEADERS, "Accept": "text/html"})
    if resp is None:
        print(f"  [DOWNLOAD FAIL] {ticker} | {doc_url}")
        return "error"

    # Save HTML to disk
    output_dir.mkdir(parents=True, exist_ok=True)
    safe_ticker    = sanitize_filename(ticker).upper()
    safe_accession = sanitize_filename(accession_number)
    html_filename  = f"10Q-{safe_ticker}-{safe_accession}.html"
    html_path      = output_dir / html_filename

    try:
        html_path.write_bytes(resp.content)
    except OSError as exc:
        print(f"  [WRITE ERROR] {html_path}: {exc}")
        return "error"

    # Save metadata
    metadata = {
        "ticker":           filing["ticker"],
        "company_name":     filing["company_name"],
        "filed_at":         filed_at,
        "accession_number": accession_number,
        "cik":              cik,
        "filing_url":       filing["filing_url"],
        "document_url":     doc_url,
        "local_html_file":  html_filename,
        "downloaded_at":    datetime.datetime.utcnow().isoformat() + "Z",
    }
    save_metadata(output_dir, metadata)

    return "downloaded"


# ---------------------------------------------------------------------------
# Main download loop
# ---------------------------------------------------------------------------

counts = {"downloaded": 0, "skipped": 0, "error": 0}
total  = len(ALL_FILINGS)

print(f"Starting download loop for {total} filings...")
print("=" * 60)

for idx, filing in enumerate(ALL_FILINGS, start=1):
    ticker  = filing.get("ticker", "UNKNOWN")
    company = filing.get("company_name", "UNKNOWN")
    date    = filing.get("filed_at", "UNKNOWN")
    accno   = filing.get("accession_number", "UNKNOWN")

    print(f"[{idx:>5}/{total}] {ticker:<10} | {date} | {company[:40]}")

    result = process_filing(filing)
    counts[result] += 1

    status_label = {"downloaded": "OK", "skipped": "SKIP", "error": "ERR"}[result]
    print(f"           -> {status_label}")

print("\nDownload loop complete.")
print(f"  Downloaded : {counts['downloaded']}")
print(f"  Skipped    : {counts['skipped']}")
print(f"  Errors     : {counts['error']}")

## Cell 6 — Summary & Statistics

Final counts, disk usage, and a listing of output directories.

In [ ]:
# Cell 6 — Summary & Statistics

import os

print("=" * 60)
print("SEC EDGAR 10-Q DOWNLOAD — FINAL SUMMARY")
print("=" * 60)
print(f"  Date range      : {START_DATE}  ->  {END_DATE}")
print(f"  Total filings   : {len(ALL_FILINGS)}")
print(f"  Downloaded      : {counts['downloaded']}")
print(f"  Skipped (dup)   : {counts['skipped']}")
print(f"  Errors          : {counts['error']}")
print()

# Disk usage of output directory
if BASE_DIR.exists():
    total_bytes = sum(
        f.stat().st_size for f in BASE_DIR.rglob("*") if f.is_file()
    )
    total_mb = total_bytes / (1024 * 1024)
    total_files = sum(1 for _ in BASE_DIR.rglob("*") if _.is_file())
    num_html    = sum(1 for _ in BASE_DIR.rglob("*.html"))
    num_meta    = sum(1 for _ in BASE_DIR.rglob("metadata.json"))

    print(f"  Output directory: {BASE_DIR}")
    print(f"  Total files     : {total_files}")
    print(f"  HTML filings    : {num_html}")
    print(f"  Metadata files  : {num_meta}")
    print(f"  Disk usage      : {total_mb:.1f} MB")
    print()

    # Show unique tickers downloaded
    tickers_downloaded = sorted(set(p.parent.parent.parent.name for p in BASE_DIR.rglob("metadata.json")))
    print(f"  Unique tickers  : {len(tickers_downloaded)}")
    if tickers_downloaded:
        preview = tickers_downloaded[:20]
        print(f"  Sample tickers  : {', '.join(preview)}{'...' if len(tickers_downloaded) > 20 else ''}")
else:
    print("  Output directory does not exist — no files were saved.")

print()
print("Done.")

In [ ]:
# Cell 45 — Individual Dashboards
# Renders a dedicated 4-panel dark-themed dashboard for each analyzed company.

for ticker, sc in all_scorecards.items():
    name = all_company_data[ticker]["name"]
    render_dashboard(sc, name, ticker)

### Comparison Dashboard

Side-by-side comparison across all analyzed companies — radar overlay, ranked scores, risk landscape, and head-to-head summary.

In [ ]:
# Cell 47 — Comparison Dashboard
# Four-panel dark-themed figure that places all companies side-by-side:
#   Panel 1 (top-left)    : Radar overlay — all companies on one polar chart
#   Panel 2 (top-right)   : Grouped horizontal bar chart ranked by avg score
#   Panel 3 (bottom-left) : Risk heatmap — severity vs company, dot-per-risk
#   Panel 4 (bottom-right): Head-to-head summary table via ax.table()

import itertools
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------
SCORE_DIMS = [
    "revenue_growth",
    "profitability",
    "margin_quality",
    "cash_position",
    "leverage_risk",
    "earnings_quality",
    "capital_efficiency",
    "overall_health",
]

DIM_LABELS = {
    "revenue_growth":     "Rev Growth",
    "profitability":      "Profitability",
    "margin_quality":     "Margin Quality",
    "cash_position":      "Cash Position",
    "leverage_risk":      "Leverage Risk",
    "earnings_quality":   "Earnings Quality",
    "capital_efficiency": "Capital Eff.",
    "overall_health":     "Overall Health",
}

COMPANY_PALETTE = [
    "#00d2ff", "#ff6b6b", "#51cf66", "#ffd43b",
    "#cc5de8", "#ff922b", "#20c997", "#da77f2",
]

RISK_CAT_COLORS = {
    "market":      "#00d2ff",
    "operational": "#ff6b6b",
    "financial":   "#ffd43b",
    "regulatory":  "#cc5de8",
    "other":       "#aaaaaa",
}

BG_COLOR   = "#1a1a2e"
PAN_COLOR  = "#16213e"
TEXT_COLOR = "#e0e0e0"
GRID_COLOR = "#2a2a4a"


# ---------------------------------------------------------------------------
# Internal helpers
# ---------------------------------------------------------------------------

def _outlook_color(outlook):
    mapping = {
        "bullish": "#51cf66",
        "neutral": "#ffd43b",
        "bearish": "#ff6b6b",
    }
    return mapping.get(str(outlook).lower(), "#aaaaaa")


def _ordinal(n):
    suffixes = {1: "st", 2: "nd", 3: "rd"}
    return "{}{}".format(n, suffixes.get(n if n < 20 else n % 10, "th"))


# ---------------------------------------------------------------------------
# Panel 1 — Radar Overlay
# ---------------------------------------------------------------------------

def _draw_radar(ax, scorecards, tickers, colors):
    num_dims = len(SCORE_DIMS)
    angles = np.linspace(0, 2 * np.pi, num_dims, endpoint=False).tolist()
    angles += angles[:1]

    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(
        [DIM_LABELS[d] for d in SCORE_DIMS],
        color=TEXT_COLOR, size=7,
    )
    ax.set_ylim(0, 10)
    ax.set_yticks([2, 4, 6, 8, 10])
    ax.set_yticklabels(["2", "4", "6", "8", "10"], color="#888888", size=6)
    ax.tick_params(axis="y", pad=2)
    ax.spines["polar"].set_color(GRID_COLOR)
    ax.set_facecolor(PAN_COLOR)
    for gridline in ax.yaxis.get_gridlines():
        gridline.set_color(GRID_COLOR)

    for ticker, color in zip(tickers, colors):
        sc = scorecards[ticker]
        values = [sc["scores"].get(d, {}).get("score", 0) for d in SCORE_DIMS]
        values += values[:1]
        ax.plot(angles, values, color=color, linewidth=1.8, linestyle="solid")
        ax.fill(angles, values, color=color, alpha=0.15)

    ax.set_title(
        "Radar Overlay",
        color=TEXT_COLOR, size=11, pad=14, fontweight="bold",
    )

    legend_handles = [
        mpatches.Patch(color=c, label=t) for t, c in zip(tickers, colors)
    ]
    ax.legend(
        handles=legend_handles,
        loc="upper right",
        bbox_to_anchor=(1.35, 1.15),
        framealpha=0.0,
        labelcolor=TEXT_COLOR,
        fontsize=7,
    )


# ---------------------------------------------------------------------------
# Panel 2 — Grouped Horizontal Bar Chart
# ---------------------------------------------------------------------------

def _draw_bar_chart(ax, scorecards, tickers, colors):
    ax.set_facecolor(PAN_COLOR)

    num_companies = len(tickers)
    bar_height = 0.7 / num_companies

    dim_avgs = {}
    for dim in SCORE_DIMS:
        scores = [scorecards[t]["scores"].get(dim, {}).get("score", 0) for t in tickers]
        dim_avgs[dim] = float(np.mean(scores))

    sorted_dims = sorted(SCORE_DIMS, key=lambda d: dim_avgs[d], reverse=True)
    y_positions = np.arange(len(sorted_dims))

    for ci, (ticker, color) in enumerate(zip(tickers, colors)):
        offsets = y_positions - (num_companies - 1) * bar_height / 2 + ci * bar_height
        values = [scorecards[ticker]["scores"].get(d, {}).get("score", 0) for d in sorted_dims]
        ax.barh(offsets, values, height=bar_height * 0.9, color=color, alpha=0.85, label=ticker)

    ax.set_yticks(y_positions)
    ax.set_yticklabels([DIM_LABELS[d] for d in sorted_dims], color=TEXT_COLOR, size=7)
    ax.set_xlim(0, 10)
    ax.set_xlabel("Score (0-10)", color=TEXT_COLOR, size=8)
    ax.tick_params(colors=TEXT_COLOR, labelsize=7)
    ax.spines["bottom"].set_color(GRID_COLOR)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color(GRID_COLOR)
    ax.xaxis.set_tick_params(colors=TEXT_COLOR)
    ax.yaxis.set_tick_params(colors=TEXT_COLOR)
    ax.xaxis.grid(True, color=GRID_COLOR, linewidth=0.5, zorder=0)
    ax.set_axisbelow(True)

    ax.set_title("Ranked Score Comparison", color=TEXT_COLOR, size=11, fontweight="bold")
    ax.legend(
        loc="lower right",
        framealpha=0.0,
        labelcolor=TEXT_COLOR,
        fontsize=7,
    )


# ---------------------------------------------------------------------------
# Panel 3 — Comparative Risk Heatmap
# ---------------------------------------------------------------------------

def _draw_risk_heatmap(ax, scorecards, tickers):
    ax.set_facecolor(PAN_COLOR)

    # Danger-zone shading above severity 3.5
    ax.axhspan(3.5, 5.3, color="#ff6b6b", alpha=0.07, zorder=0)
    ax.text(
        len(tickers) - 0.5, 4.0,
        "High Severity Zone",
        color="#ff6b6b", size=6, ha="right", va="center", alpha=0.6,
    )

    ticker_to_x = {t: i for i, t in enumerate(tickers)}

    for ticker in tickers:
        sc = scorecards[ticker]
        x_base = ticker_to_x[ticker]
        risks = sc.get("risks", [])

        # Group risks by severity for horizontal jitter
        sev_buckets = {}
        for risk in risks:
            sev = int(risk.get("severity", 1))
            sev_buckets.setdefault(sev, []).append(risk)

        for sev, sev_risks in sev_buckets.items():
            total = len(sev_risks)
            for idx, risk in enumerate(sev_risks):
                cat = str(risk.get("category", "other")).lower()
                cat_color = RISK_CAT_COLORS.get(cat, RISK_CAT_COLORS["other"])
                dot_size = sev * 200

                jitter = (idx - (total - 1) / 2) * 0.18
                xpos = x_base + jitter

                ax.scatter(
                    xpos, sev,
                    s=dot_size, color=cat_color,
                    alpha=0.80, zorder=3, edgecolors="none",
                )

                raw_name = str(risk.get("name", ""))
                label = raw_name[:20] if len(raw_name) > 20 else raw_name
                ax.annotate(
                    label,
                    xy=(xpos, sev),
                    xytext=(0, dot_size ** 0.5 / 2 + 2),
                    textcoords="offset points",
                    color=TEXT_COLOR, size=5, ha="center", va="bottom",
                    rotation=30,
                )

    ax.set_xticks(range(len(tickers)))
    ax.set_xticklabels(tickers, color=TEXT_COLOR, size=8)
    ax.set_ylim(0.3, 5.5)
    ax.set_yticks([1, 2, 3, 4, 5])
    ax.set_yticklabels(["1 Low", "2", "3 Med", "4", "5 High"], color=TEXT_COLOR, size=7)
    ax.set_ylabel("Severity", color=TEXT_COLOR, size=8)
    ax.spines["bottom"].set_color(GRID_COLOR)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color(GRID_COLOR)
    ax.yaxis.grid(True, color=GRID_COLOR, linewidth=0.5, zorder=0)
    ax.set_axisbelow(True)
    ax.tick_params(colors=TEXT_COLOR)

    ax.set_title("Risk Landscape", color=TEXT_COLOR, size=11, fontweight="bold")

    cat_handles = [
        mpatches.Patch(color=c, label=cat.capitalize())
        for cat, c in RISK_CAT_COLORS.items()
    ]
    ax.legend(
        handles=cat_handles,
        loc="upper left",
        framealpha=0.0,
        labelcolor=TEXT_COLOR,
        fontsize=6,
        ncol=2,
    )


# ---------------------------------------------------------------------------
# Panel 4 — Head-to-Head Table
# ---------------------------------------------------------------------------

def _draw_table(ax, scorecards, tickers, company_data):
    ax.set_facecolor(PAN_COLOR)
    ax.axis("off")

    row_labels = [DIM_LABELS[d] for d in SCORE_DIMS] + ["Outlook", "Overall Rank"]

    health_scores = {
        t: scorecards[t]["scores"].get("overall_health", {}).get("score", 0)
        for t in tickers
    }
    ranked = sorted(tickers, key=lambda t: health_scores[t], reverse=True)
    rank_map = {t: i + 1 for i, t in enumerate(ranked)}

    cell_text = []
    cell_colors = []

    for dim in SCORE_DIMS:
        row_text = []
        row_color_row = []
        for ticker in tickers:
            score = scorecards[ticker]["scores"].get(dim, {}).get("score", 0)
            row_text.append("{:.0f}/10".format(score))
            base_color = mcolors.to_rgb(score_color(score))
            row_color_row.append((*base_color, 0.35))
        cell_text.append(row_text)
        cell_colors.append(row_color_row)

    # Outlook row
    outlook_row_text = []
    outlook_row_colors = []
    for ticker in tickers:
        raw = scorecards[ticker].get("outlook", "neutral")
        outlook_row_text.append(raw.capitalize())
        oc = mcolors.to_rgb(_outlook_color(raw))
        outlook_row_colors.append((*oc, 0.35))
    cell_text.append(outlook_row_text)
    cell_colors.append(outlook_row_colors)

    # Overall Rank row
    rank_row_text = []
    rank_row_colors = []
    for ticker in tickers:
        rank_row_text.append(_ordinal(rank_map[ticker]))
        r = rank_map[ticker]
        if r == 1:
            rc = mcolors.to_rgb("#ffd43b")
        elif r == 2:
            rc = mcolors.to_rgb("#b0b0b0")
        elif r == 3:
            rc = mcolors.to_rgb("#cd7f32")
        else:
            rc = mcolors.to_rgb("#444466")
        rank_row_colors.append((*rc, 0.45))
    cell_text.append(rank_row_text)
    cell_colors.append(rank_row_colors)

    col_labels = []
    for t in tickers:
        name = company_data.get(t, {}).get("name", t)
        truncated = name[:10] if len(name) > 10 else name
        col_labels.append("{}\n({})".format(truncated, t))

    tbl = ax.table(
        cellText=cell_text,
        rowLabels=row_labels,
        colLabels=col_labels,
        cellColours=cell_colors,
        rowColours=[(0.09, 0.13, 0.24, 0.9)] * len(row_labels),
        colColours=[(0.05, 0.10, 0.20, 0.95)] * len(tickers),
        loc="center",
        bbox=[0, 0, 1, 1],
    )

    tbl.auto_set_font_size(False)
    tbl.set_fontsize(7)

    for (row, col), cell in tbl.get_celld().items():
        cell.set_edgecolor("none")
        cell.set_text_props(color=TEXT_COLOR, fontsize=7)
        if row == 0:
            cell.set_text_props(color="#00d2ff", fontsize=7, fontweight="bold")
            cell.set_edgecolor(GRID_COLOR)
        if col == -1:
            cell.set_text_props(color="#aaaaaa", fontsize=6.5)

    ax.set_title(
        "Head-to-Head Summary",
        color=TEXT_COLOR, size=11, fontweight="bold", pad=10,
    )


# ---------------------------------------------------------------------------
# Main public function
# ---------------------------------------------------------------------------

def render_comparison_dashboard(scorecards, company_data):
    """Render a 4-panel dark-themed comparison dashboard across all companies.

    Saves the figure to /content/comparison_dashboard.png at 150 dpi.

    Parameters
    ----------
    scorecards   : dict  {ticker: validated_scorecard}
    company_data : dict  {ticker: {name, ticker, ...}}
    """
    tickers = list(scorecards.keys())
    num_companies = len(tickers)

    palette_cycle = itertools.cycle(COMPANY_PALETTE)
    colors = [next(palette_cycle) for _ in tickers]

    # Figure setup
    fig = plt.figure(figsize=(20, 16), facecolor=BG_COLOR)
    fig.patch.set_facecolor(BG_COLOR)

    gs = fig.add_gridspec(
        2, 2,
        hspace=0.40,
        wspace=0.32,
        left=0.06, right=0.97,
        top=0.93, bottom=0.05,
    )

    ax_radar = fig.add_subplot(gs[0, 0], projection="polar")
    ax_radar.set_facecolor(PAN_COLOR)
    _draw_radar(ax_radar, scorecards, tickers, colors)

    ax_bar = fig.add_subplot(gs[0, 1])
    ax_bar.set_facecolor(PAN_COLOR)
    _draw_bar_chart(ax_bar, scorecards, tickers, colors)

    ax_risk = fig.add_subplot(gs[1, 0])
    ax_risk.set_facecolor(PAN_COLOR)
    _draw_risk_heatmap(ax_risk, scorecards, tickers)

    ax_table = fig.add_subplot(gs[1, 1])
    ax_table.set_facecolor(PAN_COLOR)
    _draw_table(ax_table, scorecards, tickers, company_data)

    fig.suptitle(
        "Comparative Financial Dashboard  \u00b7  {} Companies".format(num_companies),
        color=TEXT_COLOR,
        size=15,
        fontweight="bold",
        y=0.975,
    )

    out_path = "/content/comparison_dashboard.png"
    fig.savefig(out_path, dpi=150, bbox_inches="tight", facecolor=BG_COLOR)
    print("Comparison dashboard saved to {}".format(out_path))
    plt.show()


# ---------------------------------------------------------------------------
# Execute
# ---------------------------------------------------------------------------
company_names = {t: all_company_data[t]["name"] for t in all_scorecards}
render_comparison_dashboard(all_scorecards, all_company_data)

### Notebook Complete

This notebook performed a full pipeline from raw SEC EDGAR 10-Q filings to a structured, visual financial scorecard suite:

**What was done:**
- **Cells 1-6** — SEC EDGAR data acquisition, parsing, and download pipeline using `edgartools`.
- **Cells 7-44** — Financial data extraction, LLM-powered scorecard generation, schema validation, and individual dashboard rendering.
- **Cell 45** — Per-company dashboards rendered via `render_dashboard()`, each with four dark-themed panels (radar, bar, risk timeline, metrics grid).
- **Cell 47** — Multi-company comparison dashboard via `render_comparison_dashboard()`, saved to `/content/comparison_dashboard.png`.

**Output artifacts:**

| File | Description |
|------|-------------|
| `/content/comparison_dashboard.png` | Composite 4-panel cross-company comparison (20x16 in, 150 dpi) |
| `/content/{TICKER}_dashboard.png` | Per-company dashboard for each analyzed ticker |

**Dashboard panels:**
1. **Radar Overlay** — All companies plotted on shared polar axes; semi-transparent fills expose relative strengths.
2. **Ranked Score Bars** — Grouped horizontal bars sorted by cross-company average score, descending.
3. **Risk Landscape** — Severity-scaled scatter dots per company; danger zone highlighted above severity 3.5.
4. **Head-to-Head Table** — Score, outlook, and rank for every company in a single heatmap-colored table.

**Scorecard dimensions scored:** Revenue Growth · Profitability · Margin Quality · Cash Position · Leverage Risk · Earnings Quality · Capital Efficiency · Overall Health